# EDA и эксперименты

Этот ноутбук фиксирует разведочный анализ и экспериментальный протокол для проекта по оценке сложности английского текста. Основной исполняемый код находится в `src/text_complexity`, а ноутбук используется как читаемый журнал экспериментов.

In [ ]:
import csv
from collections import Counter
from pathlib import Path

data_path = Path('..') / 'data' / 'cefr_long_en.csv'
if not data_path.exists():
    data_path = Path('data') / 'cefr_long_en.csv'

with data_path.open(encoding='utf-8', newline='') as file:
    rows = list(csv.DictReader(file))

len(rows), rows[0]

In [ ]:
targets = [float(row['target']) for row in rows]
levels = Counter(row['cefr_level'] for row in rows)
word_lengths = [len(row['text'].split()) for row in rows]

{
    'rows': len(rows),
    'target_min': min(targets),
    'target_max': max(targets),
    'target_mean': round(sum(targets) / len(targets), 4),
    'levels': dict(levels),
    'min_words': min(word_lengths),
    'max_words': max(word_lengths),
}

## Протокол эксперимента

- Базовый ориентир: константное предсказание, равное среднему значению target на обучении.
- Финальная модель: ridge-регрессия по readability-признакам, CEFR-ориентированным лексическим признакам и TF-IDF-подобным признакам по униграммам и биграммам.
- Разбиение: 80/20 при `random_state = 42`.
- Метрики: MAE, RMSE, R2, Spearman, macro F1 и adjacent accuracy.

Запускать из корня проекта:

```bash
python -m src.text_complexity.train --config configs/config.yaml
```

In [ ]:
import json

metrics_path = Path('..') / 'artifacts' / 'metrics.json'
if not metrics_path.exists():
    metrics_path = Path('artifacts') / 'metrics.json'

if metrics_path.exists():
    json.loads(metrics_path.read_text(encoding='utf-8'))
else:
    'Сначала запустите обучение, чтобы появился artifacts/metrics.json'